# Find the Function: introduction to the data and the metric

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/majamuchowicz/golem-minihackathon/blob/main/notebook.ipynb)

This notebook gets your team from zero to a valid submission and shows you what the data really looks like.

1. Setup
2. The data: queries, candidates, pools
3. What dev/test code looks like, and how train is different
4. The metric: MRR@100
5. Your first submission: random ranking
6. Plugging in your own ranker
7. Looking at failures
8. Writing the test submission
9. Next steps and an experiment log

The task, scoring and rules are all in `docs/challenge_description.pdf`

## 1. Setup

In [ ]:
import os, sys

REPO_URL = "https://github.com/majamuchowicz/golem-minihackathon.git"

if "google.colab" in sys.modules and not os.path.exists("evaluate.py"):
    os.system(f"git clone -q {REPO_URL} kit")
    os.chdir("kit")
print("working directory:", os.getcwd())

In [ ]:
import csv, gzip, json, random, re, collections, functools
import pandas as pd

from evaluate import score

pd.set_option("display.max_colwidth", 120)
DATA = "data"

## 2. The data

| File | What it contains |
|---|---|
| `dev_queries.jsonl` | 5000 queries (`query_id`, `pool`, `query`) |
| `dev_candidates.jsonl` | 5000 functions (`candidate_id`, `pool`, `code`) |
| `dev_answers.csv` | the correct `candidate_id` for every dev query |
| `test_queries.jsonl`, `test_candidates.jsonl` | the same for the test set, **without answers** |
| `train.jsonl.gz` | 250653 matching (`query`, `code`) pairs for training |

In [ ]:
dev_q = pd.read_json(f"{DATA}/dev_queries.jsonl", lines=True)
dev_c = pd.read_json(f"{DATA}/dev_candidates.jsonl", lines=True)
dev_a = pd.read_csv(f"{DATA}/dev_answers.csv")
print(len(dev_q), "queries,", len(dev_c), "candidates")
dev_q.head(3)

### Pools

Queries and candidates are split into **pools of 1000**. Each query has exactly **one** correct function, always in its own pool, so you only rank 1000 candidates per query, not 5000. Let's check that.

In [ ]:
print("queries per pool:   ", dev_q.pool.value_counts().sort_index().tolist())
print("candidates per pool:", dev_c.pool.value_counts().sort_index().tolist())

pool_of_candidate = dict(zip(dev_c.candidate_id, dev_c.pool))
same_pool = (dev_a.candidate_id.map(pool_of_candidate) == dev_a.pool).mean()
print("share of answers in the query's own pool:", same_pool)

### A few examples

Run this cell a few times to see different random pairs and get some intuition what makes each query and function a match.

In [ ]:
code_of = dict(zip(dev_c.candidate_id, dev_c.code))
gold_of = dict(zip(dev_a.query_id, dev_a.candidate_id))

for _, row in dev_q.sample(2).iterrows():
    print("QUERY:", row.query)
    print("CORRECT FUNCTION:\n" + code_of[gold_of[row.query_id]][:600])
    print("=" * 80)

## 3. What dev/test code looks like

- Docstrings and comments are **removed**
- The function name is **`Func`**, and arguments and variables are **`arg_0`, `arg_1`, ...**
- Library calls, attribute names and string literals are kept
- Every occurrence of the function's own name became `Func`, even inside calls. A function called `show` may contain `plt.Func(...)`

In [ ]:
func_inside = dev_c[dev_c.code.str.count(r"\bFunc\b") > 1]
print(f"{len(func_inside) / len(dev_c):.0%} of candidates mention Func more than once\n")
example = dev_c[dev_c.code.str.contains(r"\.Func\(") & (dev_c.code.str.len() < 400)].code.iloc[0]
print("For example (look for .Func):\n" + example)

### Queries vary a lot in length

Most are a sentence or two, but some are whole docstrings with parameter descriptions.

In [ ]:
words = dev_q["query"].str.split().str.len()
print(words.describe(percentiles=[0.1, 0.5, 0.9, 0.99]).round(0).to_string())
long_q = dev_q[dev_q["query"].str.contains(":param|Args:|Returns:")]
print(f"\n{len(long_q) / len(dev_q):.0%} contain parameter/return sections, e.g.:\n")
print(long_q["query"].iloc[0][:500])

### Train is different

Training code keeps its real function and variable names that you won't have at test time. If you train a model, think about whether it should see names like these.

In [ ]:
with gzip.open(f"{DATA}/train.jsonl.gz", "rt") as f:
    train_sample = [json.loads(next(f)) for _ in range(1000)]
print(len(train_sample), "training pairs loaded (the full file has 250653)\n")
print("QUERY:", train_sample[0]["query"], "\n")
print(train_sample[0]["code"][:600])

## 4. The metric: MRR@100

For each query you submit a ranked list of up to 100 candidates, best first

- If the correct function is at rank r, the query scores 1/r: rank 1 gives 1.0, rank 2 gives 0.5, rank 10 gives 0.1
- If it is not in your top 100, the query scores 0

**MRR@100** is the average over all queries. A perfect system scores 1.0

In [ ]:
sample_ranks = [1, 2, 10, None]     # rank of the correct answer for 4 queries (None = not in top 100)
sample = [1 / r if r else 0 for r in sample_ranks]
print("per-query:", sample, " MRR =", sum(sample) / len(sample))

## 5. First submission: random ranking

First make sure the whole pipeline works: rank, write the file, score it. A random ranking should score about **0.005**

A ranking is a dictionary `{query_id: [candidate_id, ...]}`, best first, at most 100 per query

In [ ]:
def write_submission(ranking, path):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["query_id", "candidate_id", "rank"])
        for qid, cids in ranking.items():
            for r, cid in enumerate(cids[:100], start=1):
                w.writerow([qid, cid, r])

def candidates_by_pool(cands):
    pools = collections.defaultdict(list)
    for c in cands:
        pools[c["pool"]].append(c)
    return pools

dev_queries = dev_q.to_dict("records")
dev_cands = dev_c.to_dict("records")

random.seed(0)
pools = candidates_by_pool(dev_cands)
random_ranking = {q["query_id"]: [c["candidate_id"] for c in random.sample(pools[q["pool"]], 100)]
                  for q in dev_queries}
write_submission(random_ranking, "submissions/dev_random.csv")
score(f"{DATA}/dev_answers.csv", "submissions/dev_random.csv", f"{DATA}/dev_candidates.jsonl")

## 6. Plugging in your own ranker

`rank_with()` handles the pools and the top-100 cut-off. You only write a **scoring function**: given a query and a list of candidates, return one number per candidate (higher means more relevant)

The example below just counts the words shared by the query and the code. After you see how the pieces fit together, replace it with your own idea

In [ ]:
WORD = re.compile(r"[A-Za-z][a-z]+|[A-Z]+(?![a-z])|[0-9]+")   # splits snake_case and camelCase

@functools.lru_cache(maxsize=None)          # each text is tokenised once, then reused
def words(text):
    return frozenset(w.lower() for w in WORD.findall(text))

def shared_words_score(query, candidates):
    q = words(query["query"])
    return [len(q & words(c["code"])) for c in candidates]

def rank_with(score_fn, queries, cands, top_k=100):
    ranking = {}
    for pool, pool_cands in candidates_by_pool(cands).items():
        ids = [c["candidate_id"] for c in pool_cands]
        for q in (q for q in queries if q["pool"] == pool):
            s = score_fn(q, pool_cands)
            order = sorted(range(len(ids)), key=lambda i: -s[i])[:top_k]
            ranking[q["query_id"]] = [ids[i] for i in order]
    return ranking

In [ ]:
%%time
ranking = rank_with(shared_words_score, dev_queries, dev_cands)
write_submission(ranking, "submissions/dev_shared_words.csv")
score(f"{DATA}/dev_answers.csv", "submissions/dev_shared_words.csv", f"{DATA}/dev_candidates.jsonl")

## 7. Looking at failures

`show_failures()` works with any ranking: for a few queries whose correct answer ranked badly, it prints the query, the correct function and your top-1 prediction

In [ ]:
def gold_rank(ranking, qid):
    lst = ranking.get(qid, [])
    return lst.index(gold_of[qid]) + 1 if gold_of[qid] in lst else None

def show_failures(ranking, n=2, seed=0):
    ranks = {q: gold_rank(ranking, q) for q in gold_of}
    bad = [q for q, r in ranks.items() if r is None or r > 10]
    print(f"correct at rank 1: {sum(r == 1 for r in ranks.values()) / len(ranks):.1%}   "
          f"not in top 100: {sum(r is None for r in ranks.values()) / len(ranks):.1%}\n")
    query_text = dict(zip(dev_q.query_id, dev_q["query"]))
    for qid in random.Random(seed).sample(bad, n):
        print("QUERY:", query_text[qid][:300])
        print(f"\nCORRECT FUNCTION (your rank: {ranks[qid] or '>100'}):\n" + code_of[gold_of[qid]][:500])
        print("\nYOUR TOP-1:\n" + code_of[ranking[qid][0]][:500])
        print("=" * 80)

show_failures(ranking)

## 8. Writing the test submission

The test answers are hidden, and we will score it once at the end with the same `evaluate.py`. Feel free to check the format of your submission beforehand

In [ ]:
%%time
test_queries = pd.read_json(f"{DATA}/test_queries.jsonl", lines=True).to_dict("records")
test_cands = pd.read_json(f"{DATA}/test_candidates.jsonl", lines=True).to_dict("records")

test_ranking = rank_with(shared_words_score, test_queries, test_cands)   # <- use your best ranker here
write_submission(test_ranking, "submissions/submission_test.csv")

sub = pd.read_csv("submissions/submission_test.csv")
test_pool = {c["candidate_id"]: c["pool"] for c in test_cands}
query_pool = {q["query_id"]: q["pool"] for q in test_queries}
assert list(sub.columns) == ["query_id", "candidate_id", "rank"]
assert sub.query_id.nunique() == len(test_queries), "every test query needs a ranking"
assert sub["rank"].between(1, 100).all()
assert not sub.duplicated(["query_id", "rank"]).any() and not sub.duplicated(["query_id", "candidate_id"]).any()
assert (sub.candidate_id.map(test_pool) == sub.query_id.map(query_pool)).all(), "candidates must come from the query's pool"
print("submission format OK:", sub.shape)